# MOSS-Image · T4 ComfyUI 프론트
[최신 Colab 노트북](https://github.com/HisameOgasahara/tmp_moss#readme)

**T4 GPU 선택 → 1~3번 실행 → ComfyUI 열기 → 워크플로 불러오기 → Run.**

- N1-lora8+spectrum: INT8 본체 / W4A8 인코더 / 8스텝 LoRA / Spectrum Manual.
- 기본값: 1024×1024 · batch 1 · Euler · sgm_uniform · 8스텝 · CFG 1.
- 프롬프트·해상도·시드·생성 설정은 ComfyUI 화면에서 변경합니다. 생성 이미지는 Save Image 노드에서 저장합니다.
- 워크플로는 서버의 Workflows 목록에 등록되며 4번 셀에서도 JSON을 내려받을 수 있습니다.
- 같은 Colab 런타임의 기존 `/content/moss_t4` 모델 캐시를 재사용합니다.

터널 주소를 아는 사람은 ComfyUI에 접속할 수 있습니다. 사용 후 5번 셀의 `STOP`을 켜고 실행하면 연결이 종료됩니다.


In [ ]:
#@title 1. T4 · ComfyUI · Spectrum 설치
import sys, os, json, time, re, shutil, hashlib, subprocess, urllib.request, urllib.error
from pathlib import Path
from IPython.display import display, HTML

ROOT = Path('/content/moss_t4')
COMFY = ROOT/'ComfyUI'
VENV = ROOT/'venv'
PY = str(VENV/'bin/python')
PORT = 8188
LOCAL_URL = f'http://127.0.0.1:{PORT}'
COMFY_REV = 'bc5f880dbdd43e426077f42ccec53bb577ffee9f'
SPECTRUM_REV = 'd3f7e243ea2db6cc3ff26f42b935b167f938b599'
MODEL_REV = '24147db865fcdc207270f844777587223fed8d6e'
LORA_REV = 'f95df83b5e90ae91ea412434b3f9b7f1c8c6314f'
ROOT.mkdir(parents=True,exist_ok=True)
INSTALL_LOG = ROOT/'frontend-install.log'
INSTALL_LOG.write_text('',encoding='utf-8')
gpu = subprocess.check_output(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],text=True).strip()
if 'T4' not in gpu:
    raise RuntimeError(f'T4 GPU를 선택하세요. 현재 GPU: {gpu}')
print('GPU:',gpu)

def run(command, **kwargs):
    command = [str(x) for x in command]
    lines = []
    with INSTALL_LOG.open('a',encoding='utf-8') as log:
        log.write('> '+' '.join(command)+chr(10))
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1,**kwargs) as process:
            for line in process.stdout:
                print(line,end='',flush=True)
                log.write(line)
                log.flush()
                lines.append(line)
            code = process.wait()
    if code:
        raise RuntimeError(f'설치 실패 (exit {code}) · {INSTALL_LOG}'+chr(10)+''.join(lines[-40:]))

def fetch(url):
    request = urllib.request.Request(url,headers={'User-Agent':'moss-t4-comfyui'})
    with urllib.request.urlopen(request,timeout=120) as response:
        return response.read()

if shutil.which('uv') is None:
    run([sys.executable,'-m','pip','install','-q','uv>=0.9.27'])
if '--excludes' not in subprocess.check_output(['uv','pip','install','--help'],text=True):
    run([sys.executable,'-m','pip','install','-q','--upgrade','uv>=0.9.27'])
if not Path(PY).exists():
    run(['uv','venv','--system-site-packages','--python',sys.executable,VENV])
if not (COMFY/'.git').exists():
    run(['git','clone','--quiet','https://github.com/Comfy-Org/ComfyUI.git',COMFY])
run(['git','-C',COMFY,'checkout','--quiet',COMFY_REV])
# Colab의 기존 torch/CUDA 패키지를 상속합니다.
runtime_code = """
import importlib.metadata as m, json
packages = {}
for dist in m.distributions():
    name = dist.metadata['Name'].lower().replace('_','-')
    if name in ('torch','torchvision','torchaudio','triton') or name.startswith('nvidia-'):
        packages[name] = dist.version
print(json.dumps(packages))
"""
runtime_packages = json.loads(subprocess.check_output([sys.executable,'-c',runtime_code],text=True))
excludes = ROOT/'colab-runtime-excludes.txt'
excludes.write_text(chr(10).join(sorted(runtime_packages))+chr(10),encoding='utf-8')
run(['uv','pip','install','--python',PY,'--excludes',excludes,'-r',COMFY/'requirements.txt','huggingface_hub'])
addon = COMFY/'custom_nodes'/'ComfyUI-Spectrum-MOSS'
if not (addon/'.git').exists():
    run(['git','clone','--quiet','https://github.com/Rkkss/ComfyUI-Spectrum-MOSS.git',addon])
run(['git','-C',addon,'checkout','--quiet',SPECTRUM_REV])
print('ComfyUI · Spectrum 설치 완료')


In [ ]:
#@title 2. 선정 모델 · 워크플로 · cloudflared 다운로드
resources = [
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='diffusion_models/moss_image_sft_int8_convrot.safetensors',
         target=str(COMFY/'models/diffusion_models/moss_image_sft_int8_convrot.safetensors')),
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='text_encoders/qwen3.5_9b_w4a8_gptq.safetensors',
         target=str(COMFY/'models/text_encoders/qwen3.5_9b_w4a8_gptq.safetensors')),
    dict(repo='Comfy-Org/MOSS-Image',revision=MODEL_REV,
         filename='vae/flux2-vae_small.safetensors',
         target=str(COMFY/'models/vae/flux2-vae_small.safetensors')),
    dict(repo='lodestones/MOSS-Image-SFT-8-steps-dmd',revision=LORA_REV,filename='lora.safetensors',
         target=str(COMFY/'models/loras/moss_sft_8step_lora.safetensors')),
]
manifest = ROOT/'frontend-downloads.json'
manifest.write_text(json.dumps(resources,indent=2),encoding='utf-8')
download_code = """
import json, sys
from pathlib import Path
from huggingface_hub import hf_hub_download
for item in json.loads(Path(sys.argv[1]).read_text()):
    print('Download:',item['filename'],flush=True)
    source = hf_hub_download(item['repo'],item['filename'],revision=item['revision'],cache_dir=sys.argv[2])
    target = Path(item['target'])
    target.parent.mkdir(parents=True,exist_ok=True)
    if target.is_symlink():
        target.unlink()
    if not target.exists():
        target.symlink_to(source)
"""
run([PY,'-c',download_code,manifest,ROOT/'hf-cache'])

WORKFLOW_JSON = '{\n  "last_node_id": 11,\n  "last_link_id": 11,\n  "nodes": [\n    {\n      "id": 1,\n      "type": "UNETLoader",\n      "title": "MOSS INT8 본체",\n      "pos": [\n        40,\n        40\n      ],\n      "size": [\n        340,\n        110\n      ],\n      "flags": {},\n      "order": 0,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            1\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "UNETLoader"\n      },\n      "widgets_values": [\n        "moss_image_sft_int8_convrot.safetensors",\n        "default"\n      ]\n    },\n    {\n      "id": 2,\n      "type": "LoraLoaderModelOnly",\n      "title": "8스텝 LoRA",\n      "pos": [\n        40,\n        210\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 1,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 1\n        }\n      ],\n      "outputs": [\n        {\n          "name": "MODEL",\n          "type": "MODEL",\n          "links": [\n            2\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "LoraLoaderModelOnly"\n      },\n      "widgets_values": [\n        "moss_sft_8step_lora.safetensors",\n        1.0\n      ]\n    },\n    {\n      "id": 3,\n      "type": "SpectrumMOSSImageManual",\n      "title": "Spectrum · 검증 설정",\n      "pos": [\n        440,\n        40\n      ],\n      "size": [\n        350,\n        490\n      ],\n      "flags": {},\n      "order": 2,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 2\n        }\n      ],\n      "outputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "links": [\n            3\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "SpectrumMOSSImageManual"\n      },\n      "widgets_values": [\n        2,\n        0,\n        2.0,\n        0.75,\n        2,\n        2,\n        1,\n        0.1,\n        0.5,\n        "main_device",\n        true,\n        false,\n        true\n      ]\n    },\n    {\n      "id": 4,\n      "type": "CLIPLoader",\n      "title": "Qwen W4A8 인코더",\n      "pos": [\n        40,\n        420\n      ],\n      "size": [\n        340,\n        130\n      ],\n      "flags": {},\n      "order": 3,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "CLIP",\n          "type": "CLIP",\n          "links": [\n            4,\n            5\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPLoader"\n      },\n      "widgets_values": [\n        "qwen3.5_9b_w4a8_gptq.safetensors",\n        "moss_image",\n        "default"\n      ]\n    },\n    {\n      "id": 5,\n      "type": "VAELoader",\n      "title": "FLUX.2 small VAE",\n      "pos": [\n        840,\n        670\n      ],\n      "size": [\n        330,\n        80\n      ],\n      "flags": {},\n      "order": 4,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "VAE",\n          "type": "VAE",\n          "links": [\n            10\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAELoader"\n      },\n      "widgets_values": [\n        "flux2-vae_small.safetensors"\n      ]\n    },\n    {\n      "id": 6,\n      "type": "CLIPTextEncode",\n      "title": "양성 프롬프트",\n      "pos": [\n        440,\n        590\n      ],\n      "size": [\n        350,\n        180\n      ],\n      "flags": {},\n      "order": 5,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 4\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            6\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        "A studio photograph of a red ceramic teapot on a wooden table, soft window light."\n      ]\n    },\n    {\n      "id": 7,\n      "type": "CLIPTextEncode",\n      "title": "음성 프롬프트",\n      "pos": [\n        440,\n        840\n      ],\n      "size": [\n        350,\n        120\n      ],\n      "flags": {},\n      "order": 6,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "clip",\n          "type": "CLIP",\n          "link": 5\n        }\n      ],\n      "outputs": [\n        {\n          "name": "CONDITIONING",\n          "type": "CONDITIONING",\n          "links": [\n            7\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "CLIPTextEncode"\n      },\n      "widgets_values": [\n        ""\n      ]\n    },\n    {\n      "id": 8,\n      "type": "EmptyFlux2LatentImage",\n      "title": "해상도 · batch 1",\n      "pos": [\n        840,\n        450\n      ],\n      "size": [\n        330,\n        120\n      ],\n      "flags": {},\n      "order": 7,\n      "mode": 0,\n      "inputs": [],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            8\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "EmptyFlux2LatentImage"\n      },\n      "widgets_values": [\n        1024,\n        1024,\n        1\n      ]\n    },\n    {\n      "id": 9,\n      "type": "KSampler",\n      "title": "Euler · 8스텝 · CFG 1",\n      "pos": [\n        840,\n        40\n      ],\n      "size": [\n        330,\n        320\n      ],\n      "flags": {},\n      "order": 8,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "model",\n          "type": "MODEL",\n          "link": 3\n        },\n        {\n          "name": "positive",\n          "type": "CONDITIONING",\n          "link": 6\n        },\n        {\n          "name": "negative",\n          "type": "CONDITIONING",\n          "link": 7\n        },\n        {\n          "name": "latent_image",\n          "type": "LATENT",\n          "link": 8\n        }\n      ],\n      "outputs": [\n        {\n          "name": "LATENT",\n          "type": "LATENT",\n          "links": [\n            9\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "KSampler"\n      },\n      "widgets_values": [\n        42,\n        "randomize",\n        8,\n        1.0,\n        "euler",\n        "sgm_uniform",\n        1.0\n      ]\n    },\n    {\n      "id": 10,\n      "type": "VAEDecode",\n      "title": "VAE 디코딩",\n      "pos": [\n        1240,\n        40\n      ],\n      "size": [\n        230,\n        100\n      ],\n      "flags": {},\n      "order": 9,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "samples",\n          "type": "LATENT",\n          "link": 9\n        },\n        {\n          "name": "vae",\n          "type": "VAE",\n          "link": 10\n        }\n      ],\n      "outputs": [\n        {\n          "name": "IMAGE",\n          "type": "IMAGE",\n          "links": [\n            11\n          ],\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "VAEDecode"\n      },\n      "widgets_values": []\n    },\n    {\n      "id": 11,\n      "type": "SaveImage",\n      "title": "생성 이미지",\n      "pos": [\n        1540,\n        40\n      ],\n      "size": [\n        470,\n        500\n      ],\n      "flags": {},\n      "order": 10,\n      "mode": 0,\n      "inputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "link": 11\n        }\n      ],\n      "outputs": [\n        {\n          "name": "images",\n          "type": "IMAGE",\n          "links": null,\n          "slot_index": 0\n        }\n      ],\n      "properties": {\n        "Node name for S&R": "SaveImage"\n      },\n      "widgets_values": [\n        "MOSS_T4"\n      ]\n    }\n  ],\n  "links": [\n    [\n      1,\n      1,\n      0,\n      2,\n      0,\n      "MODEL"\n    ],\n    [\n      2,\n      2,\n      0,\n      3,\n      0,\n      "MODEL"\n    ],\n    [\n      3,\n      3,\n      0,\n      9,\n      0,\n      "MODEL"\n    ],\n    [\n      4,\n      4,\n      0,\n      6,\n      0,\n      "CLIP"\n    ],\n    [\n      5,\n      4,\n      0,\n      7,\n      0,\n      "CLIP"\n    ],\n    [\n      6,\n      6,\n      0,\n      9,\n      1,\n      "CONDITIONING"\n    ],\n    [\n      7,\n      7,\n      0,\n      9,\n      2,\n      "CONDITIONING"\n    ],\n    [\n      8,\n      8,\n      0,\n      9,\n      3,\n      "LATENT"\n    ],\n    [\n      9,\n      9,\n      0,\n      10,\n      0,\n      "LATENT"\n    ],\n    [\n      10,\n      5,\n      0,\n      10,\n      1,\n      "VAE"\n    ],\n    [\n      11,\n      10,\n      0,\n      11,\n      0,\n      "IMAGE"\n    ]\n  ],\n  "groups": [],\n  "config": {},\n  "extra": {\n    "ds": {\n      "scale": 0.7,\n      "offset": [\n        60,\n        80\n      ]\n    }\n  },\n  "version": 0.4\n}\n'
workflow = json.loads(WORKFLOW_JSON)
WORKFLOW_FILE = ROOT/'MOSS_T4_LoRA8_Spectrum.json'
WORKFLOW_FILE.write_text(WORKFLOW_JSON,encoding='utf-8')
workflow_library = COMFY/'user'/'default'/'workflows'/WORKFLOW_FILE.name
workflow_library.parent.mkdir(parents=True,exist_ok=True)
workflow_library.write_text(WORKFLOW_JSON,encoding='utf-8')

CLOUDFLARED_VERSION = '2026.10.3'
CLOUDFLARED_SHA256 = '053b80b0a23031a038fdbf3c073f59f82b8f1f152aa67971d7383ffafc0dbab0'
CLOUDFLARED = ROOT/'cloudflared'
if not CLOUDFLARED.exists() or hashlib.sha256(CLOUDFLARED.read_bytes()).hexdigest()!=CLOUDFLARED_SHA256:
    binary = fetch(f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-amd64')
    if hashlib.sha256(binary).hexdigest()!=CLOUDFLARED_SHA256:
        raise RuntimeError('cloudflared 다운로드의 SHA256이 일치하지 않습니다.')
    CLOUDFLARED.write_bytes(binary)
CLOUDFLARED.chmod(0o755)
(ROOT/'frontend-environment.json').write_text(json.dumps(dict(
    comfy_revision=COMFY_REV,spectrum_revision=SPECTRUM_REV,model_revision=MODEL_REV,lora_revision=LORA_REV,
    cloudflared_version=CLOUDFLARED_VERSION,gpu=gpu,colab_runtime_packages=runtime_packages),indent=2),encoding='utf-8')
print('모델 4개 · 워크플로 · cloudflared 준비 완료')


In [ ]:
#@title 3. ComfyUI 시작 · cloudflared 접속 주소
COMFY_LOG = ROOT/'comfyui-frontend.log'
TUNNEL_LOG = ROOT/'cloudflared.log'

def log_tail(path):
    return chr(10).join(path.read_text(encoding='utf-8',errors='replace').splitlines()[-60:]) if path.exists() else ''

def start_logged(command,path):
    with path.open('w') as log:
        return subprocess.Popen([str(x) for x in command],cwd=COMFY,
            stdout=log,stderr=subprocess.STDOUT,start_new_session=True)

if 'comfy_process' not in globals() or comfy_process.poll() is not None:
    comfy_process = start_logged([PY,COMFY/'main.py','--listen','127.0.0.1','--port',PORT,'--disable-auto-launch'],COMFY_LOG)
deadline = time.monotonic()+180
while True:
    if comfy_process.poll() is not None:
        raise RuntimeError('ComfyUI 시작 실패'+chr(10)+log_tail(COMFY_LOG))
    try:
        with urllib.request.urlopen(LOCAL_URL+'/object_info',timeout=10) as response:
            object_info = json.load(response)
        break
    except (urllib.error.URLError,TimeoutError):
        if time.monotonic()>deadline:
            raise RuntimeError('ComfyUI 시작 시간 초과'+chr(10)+log_tail(COMFY_LOG))
        time.sleep(1)
missing = {node['type'] for node in workflow['nodes']}-set(object_info)
if missing:
    raise RuntimeError(f'필수 노드 로딩 실패: {sorted(missing)}'+chr(10)+log_tail(COMFY_LOG))
print('ComfyUI · 필수 노드 준비 완료')

if 'tunnel_process' not in globals() or tunnel_process.poll() is not None:
    tunnel_process = start_logged([CLOUDFLARED,'--no-autoupdate','tunnel',
        '--url',LOCAL_URL,'--protocol','http2'],TUNNEL_LOG)
deadline = time.monotonic()+120
while True:
    text = TUNNEL_LOG.read_text(encoding='utf-8',errors='replace') if TUNNEL_LOG.exists() else ''
    match = re.search(r'https://[a-z0-9-]+[.]trycloudflare[.]com',text)
    if match and 'Registered tunnel connection' in text:
        TUNNEL_URL = match.group(0)
        break
    if tunnel_process.poll() is not None:
        raise RuntimeError('cloudflared 시작 실패'+chr(10)+log_tail(TUNNEL_LOG))
    if time.monotonic()>deadline:
        tunnel_process.terminate()
        raise RuntimeError('cloudflared 연결 시간 초과'+chr(10)+log_tail(TUNNEL_LOG))
    time.sleep(1)
display(HTML(f'<h3><a href="{TUNNEL_URL}" target="_blank" rel="noopener noreferrer">ComfyUI 열기</a></h3>'))
print(TUNNEL_URL)
print('Workflows 목록에서 MOSS_T4_LoRA8_Spectrum을 열거나, 4번 셀의 JSON을 화면에 드래그하세요.')


In [ ]:
#@title 4. 워크플로 JSON 다운로드
from google.colab import files
files.download(str(WORKFLOW_FILE))


In [ ]:
#@title 5. ComfyUI · 터널 종료
STOP = False #@param {type:'boolean'}
if STOP:
    import signal
    for key in ('tunnel_process','comfy_process'):
        process = globals().get(key)
        if process is not None and process.poll() is None:
            os.killpg(process.pid,signal.SIGTERM)
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid,signal.SIGKILL)
                process.wait()
    print('ComfyUI와 터널을 종료했습니다.')
